# Paged Optimizers & CUDA Unified Memory Systems

Even with a 4-bit base model and 16-bit LoRA adapters, fine-tuning can still trigger sudden Out-Of-Memory (OOM) crashes during long-sequence activation spikes or optimizer update steps. QLoRA introduces Paged Optimizers to eliminate memory spikes.

---

## The VRAM Memory Spike Problem

During standard training, GPU VRAM usage is not static—it experiences sharp transient spikes:

**VRAM Allocation over Time:**

```
100% VRAM ┌──────────────────────────────────────────────┐
          │                                  ▲  (OOM Crash!)
          │                                 ╱│╲
          │                                ╱ │ ╲  <-- Activation Spike on Long Sequence
          │    ┌──────────────────────────┐  │  └───────────────
          │    │ Steady-State Base Weights│  │
 0% VRAM  └───┴──────────────────────────┴──┴─────────────────► Step Execution
```

**Primary Causes of VRAM Spikes:**

* **Activation Memory Allocation:** Processing a batch with a longer-than-average prompt sequence length causes activation memory to jump proportionally
* **Optimizer State Allocations:** During the optimizer step (`optimizer.step()`), temporary memory buffers are allocated to calculate gradient updates and momentum terms for trainable parameters

Without paging, if peak memory exceeds total physical VRAM for even a few milliseconds, CUDA throws an unrecoverable `torch.cuda.OutOfMemoryError`.

---

## CUDA Unified Memory & Page Eviction Mechanics

Paged Optimizers leverage NVIDIA CUDA Unified Memory (CUDA UVM). UVM creates a single, contiguous virtual address space that maps across both physical GPU VRAM and host CPU System RAM.

**Memory Page Swapping Flow:**

```
                        [ Unified Virtual Memory Space ]
                                       │
                    ┌──────────────────┴──────────────────────┐
                    ▼                                         ▼
          [ Physical GPU VRAM ]                    [ Host CPU System RAM ]
     ┌─────────────────────────────┐          ┌─────────────────────────────┐
     │ Active Forward/Backward     │          │ Non-Active Optimizer States │
     │ Tensors & Activations       │          │ (paged out during pass)     │
     └──────────────┬──────────────┘          └──────────────┬──────────────┘
                    │                                        │
                    │  ◄── Page Fault Eviction / Page-In ────┘
```

**Page Allocation:**
* Optimizer states (e.g., AdamW FP32 momentum $m_t$ and variance $v_t$) are allocated inside CUDA UVM paged memory pointers (`cudaMallocManaged`)

**Page Eviction (GPU → CPU):**
* During the forward and backward passes, when GPU VRAM fills up due to activation allocations, the CUDA driver automatically evicts inactive optimizer memory pages over the PCIe bus into host CPU System RAM

**Page Restoration (CPU → GPU):**
* When the backward pass completes and `optimizer.step()` begins executing, the evicted pages are swapped back from CPU RAM into GPU VRAM as needed for parameter updates

## Production Tradeoffs: Paged vs. Fused Optimizers

While Paged Optimizers eliminate OOM crashes, they introduce a distinct performance penalty.

| Metric / Dimension | Standard AdamW | Paged AdamW (paged_adamw_8bit / 32bit) | Fused AdamW (fused=True) |
|---|---|---|---|
| **OOM Protection** | None (Fails on memory spike) | Maximum (Swaps memory to CPU RAM) | None (Fails on memory spike) |
| **PCIe Bus Overhead** | Zero | High (Frequent page transfers over PCIe) | Zero |
| **Training Speed** | Baseline | ~10-25% Slower (PCIe transfer bottleneck) | ~30-40% Faster (Kernel fusion) |
| **Best Use Case** | Small models with high VRAM headroom | Large models on tight VRAM GPUs (e.g., 70B on 48GB GPU) | Fixed, well-calibrated sequence lengths with VRAM headroom |

## Optimizer Selection Guidelines

* **Use `paged_adamw_8bit` or `paged_adamw_32bit`** when fine-tuning large models where peak VRAM usage touches $>90\%$ of total GPU memory
* **Switch to fused AdamW** if VRAM usage sits comfortably below $80\%$ to eliminate PCIe paging latencies and maximize token throughput